In [3]:
# SECTION 0 – INSTALL

!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q faiss-cpu
%pip install -q ultralytics
!pip install -q Pillow opencv-python-headless

  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


In [15]:
# SECTION 1 – IMPORTS & CONFIG
import os, json, gc, pickle, random, time, warnings
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

import clip
import faiss
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_BASE   = Path('/kaggle/input/datasets/venkat96r/vr-inshop-dataset/img')
IMG_ROOT       = DATASET_BASE
PARTITION_FILE = DATASET_BASE / 'list_eval_partition.txt'
YOLO_WEIGHTS   = '/kaggle/input/datasets/venkat96r/vr-final-yolo/best_yolo_fine_tuned_without_augmenatation.pt'
CAPTION_CACHE  = Path('/kaggle/input/datasets/venkatrrr/caption-cache/caption_cache.json')
OUT_DIR        = Path('/kaggle/working')

# ── Seeds (add / remove freely) ───────────────────────────────────────────────
# 1 seed ≈ 2.1h | 2 seeds ≈ 4.2h+prep | 3 seeds ≈ 6.3h+prep (all on T4×2)
SEEDS = [2023085]   # e.g. [2023085, 2023102, 2023613]

# ── Hyperparams ───────────────────────────────────────────────────────────────
CLIP_MODEL       = 'ViT-B/32'
CLIP_DIM         = 512
BATCH_SIZE       = 256        # safe for T4×2 with partial grad + AMP
LR               = 1e-5
NUM_EPOCHS       = 5          # ~25 min/epoch → ~2.1h for 1 seed
TEMPERATURE      = 0.07       # fixed InfoNCE temperature (original CLIP)
UNFREEZE_LAST_N  = 4          # unfreeze last N visual transformer ResBlocks
WEIGHT_DECAY     = 0.01
TOP_K            = [5, 10, 15]
ALPHA_VALUES     = [0.7, 0.3, 0.6]
FAISS_M          = 32
FAISS_EF         = 200
NUM_WORKERS      = 4
CONF_THRESH      = 0.25       # YOLO confidence threshold

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_GPUS = torch.cuda.device_count()

print(f"Device: {DEVICE} | GPUs: {NUM_GPUS}")
print(f"Seeds: {SEEDS} | Epochs per seed: {NUM_EPOCHS}")
total_min = 35 + len(SEEDS) * NUM_EPOCHS * 25
print(f"Estimated total time: ~{total_min} min (~{total_min/60:.1f}h)")



Device: cuda | GPUs: 2
Seeds: [2023085] | Epochs per seed: 5
Estimated total time: ~160 min (~2.7h)


In [5]:
# =============================================================================
# SECTION 2 – SEED UTILITY
# =============================================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# SECTION 3 – PARSE DATASET SPLIT
# =============================================================================
def parse_partition(partition_file: Path, img_root: Path):
    train, query, gallery = [], [], []
    with open(partition_file, 'r') as f:
        lines = f.read().splitlines()
    for line in lines[2:]:
        parts = line.split()
        if len(parts) < 3:
            continue
        rel_path, item_id, split = parts[0], parts[1], parts[2].lower().strip()
        abs_path = img_root / rel_path
        entry = (str(abs_path), item_id)
        if split == 'train':
            train.append(entry)
        elif split == 'query':
            query.append(entry)
        elif split == 'gallery':
            gallery.append(entry)
    return train, query, gallery


train_data, query_data, gallery_data = parse_partition(PARTITION_FILE, IMG_ROOT)
print(f"Train: {len(train_data)} | Query: {len(query_data)} | Gallery: {len(gallery_data)}")


Train: 25882 | Query: 14218 | Gallery: 12612


In [ ]:
# =============================================================================
# SECTION 4 – YOLO BBOX CACHE  (precomputed once, ~25-30 min)
# =============================================================================
BBOX_CACHE_PATH = OUT_DIR / 'bbox_cache_C.pkl'

def build_bbox_cache(all_data, yolo_weights, device, conf_thresh=0.25):
    """Run YOLO on every image and cache bounding boxes."""
    yolo = YOLO(yolo_weights)
    yolo.to(device)
    cache = {}
    for path, _ in tqdm(all_data, desc='YOLO bbox cache'):
        try:
            results = yolo(path, conf=conf_thresh, verbose=False)
            boxes = results[0].boxes
            if boxes is not None and len(boxes) > 0:
                best = boxes.conf.argmax().item()
                x1, y1, x2, y2 = boxes.xyxy[best].cpu().numpy().astype(int)
                cache[path] = (int(x1), int(y1), int(x2), int(y2))
            else:
                cache[path] = None
        except Exception:
            cache[path] = None
    del yolo
    gc.collect()
    torch.cuda.empty_cache()
    return cache


if BBOX_CACHE_PATH.exists():
    print(f"Loading existing bbox cache from {BBOX_CACHE_PATH}")
    with open(BBOX_CACHE_PATH, 'rb') as f:
        bbox_cache = pickle.load(f)
    print(f"  Loaded {len(bbox_cache):,} bbox entries")
else:
    print("Building YOLO bbox cache (runs once)…")
    t0 = time.time()
    all_data = train_data + query_data + gallery_data
    bbox_cache = build_bbox_cache(all_data, YOLO_WEIGHTS, DEVICE, CONF_THRESH)
    with open(BBOX_CACHE_PATH, 'wb') as f:
        pickle.dump(bbox_cache, f)
    print(f"  Done in {(time.time()-t0)/60:.1f} min | Saved → {BBOX_CACHE_PATH}")


def crop_with_cache(path: str, bbox_cache: dict) -> Image.Image:
    """Return YOLO-cropped PIL image using pre-computed bbox."""
    img = Image.open(path).convert('RGB')
    bbox = bbox_cache.get(path)
    if bbox is not None:
        x1, y1, x2, y2 = bbox
        x1, y1 = max(0, x1), max(0, y1)
        crop = img.crop((x1, y1, x2, y2))
        if crop.width > 10 and crop.height > 10:
            return crop
    return img


In [6]:
# =============================================================================
# SECTION 5 – CAPTION CACHE
# =============================================================================
print(f"Loading captions from {CAPTION_CACHE}")
with open(CAPTION_CACHE, 'r') as f:
    caption_cache: dict = json.load(f)
print(f"  Loaded {len(caption_cache):,} captions")

all_paths = set(p for p, _ in query_data + gallery_data + train_data)
missing   = all_paths - set(caption_cache.keys())
print(f"  Coverage: {len(all_paths)-len(missing)}/{len(all_paths)} | Missing: {len(missing)}")


Loading captions from /kaggle/input/datasets/venkatrrr/caption-cache/caption_cache.json
  Loaded 26,830 captions
  Coverage: 26830/52712 | Missing: 25882


In [ ]:
# =============================================================================
# SECTION 6 – CLIP SETUP: FREEZE ALL → UNFREEZE LAST N VISUAL BLOCKS
# =============================================================================
def setup_clip_for_finetuning(clip_model_name, device, unfreeze_last_n, num_gpus):
    """Load CLIP, freeze everything, unfreeze last N visual ResBlocks + ln_post."""
    model, preprocess = clip.load(clip_model_name, device=device)
    model = model.float()   # fp32 weights; AMP handles compute precision

    # Freeze all
    for p in model.parameters():
        p.requires_grad_(False)

    # Unfreeze last N visual transformer ResBlocks
    vis = model.visual
    n_blocks = len(vis.transformer.resblocks)
    for i in range(n_blocks - unfreeze_last_n, n_blocks):
        for p in vis.transformer.resblocks[i].parameters():
            p.requires_grad_(True)

    # Unfreeze visual projection head (ln_post + proj)
    for p in vis.ln_post.parameters():
        p.requires_grad_(True)
    if vis.proj is not None:
        vis.proj.requires_grad_(True)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {trainable/1e6:.2f}M / {total/1e6:.2f}M")

    if num_gpus > 1:
        model = nn.DataParallel(model)
        print(f"  Wrapped in DataParallel across {num_gpus} GPUs")

    return model, preprocess



In [7]:
# =============================================================================
# SECTION 7 – DATASET & DATALOADER
# =============================================================================
class FashionTrainDataset(Dataset):
    def __init__(self, data, caption_cache, bbox_cache, clip_preprocess):
        self.data           = data
        self.caption_cache  = caption_cache
        self.bbox_cache     = bbox_cache
        self.clip_preprocess = clip_preprocess

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, _ = self.data[idx]
        img  = crop_with_cache(path, self.bbox_cache)
        img  = self.clip_preprocess(img)
        text = self.caption_cache.get(path, '')
        return img, text


def collate_fn(batch):
    imgs, texts = zip(*batch)
    imgs = torch.stack(imgs, dim=0)
    return imgs, list(texts)


In [8]:
# =============================================================================
# SECTION 8 – INFONCE LOSS
# =============================================================================
def infonce_loss(img_feat, txt_feat, temperature=0.07):
    """Symmetric InfoNCE with in-batch negatives (CLIP-style)."""
    # img_feat, txt_feat: (B, D) L2-normalised
    B = img_feat.shape[0]
    logits_i2t = img_feat @ txt_feat.T / temperature   # (B, B)
    logits_t2i = txt_feat @ img_feat.T / temperature   # (B, B)
    labels = torch.arange(B, device=img_feat.device)
    loss = (F.cross_entropy(logits_i2t, labels) + F.cross_entropy(logits_t2i, labels)) / 2
    return loss


In [9]:
# =============================================================================
# SECTION 9 – EMBEDDING EXTRACTION (for eval / final indexing)
# =============================================================================
@torch.no_grad()
def extract_embeddings(data, model, preprocess, bbox_cache, caption_cache,
                        clip_dim, batch_size, device, alpha, desc='Embed'):
    """Extract fused (alpha * visual + (1-alpha) * text) embeddings."""
    raw_model = model.module if hasattr(model, 'module') else model
    paths = [p for p, _ in data]
    ids   = [i for _, i in data]
    N     = len(paths)

    vis_embs = np.zeros((N, clip_dim), dtype=np.float32)
    txt_embs = np.zeros((N, clip_dim), dtype=np.float32)

    for start in tqdm(range(0, N, batch_size), desc=desc, leave=False):
        bp = paths[start: start + batch_size]
        bs = len(bp)

        imgs = []
        for p in bp:
            try:
                img = crop_with_cache(p, bbox_cache)
                imgs.append(preprocess(img))
            except Exception:
                imgs.append(torch.zeros(3, 224, 224))

        img_tensor = torch.stack(imgs).to(device)
        captions   = [caption_cache.get(p, '') for p in bp]
        tokens     = clip.tokenize(captions, truncate=True).to(device)

        with autocast():
            v = raw_model.encode_image(img_tensor).float()
            t = raw_model.encode_text(tokens).float()

        v = v / v.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        t = t / t.norm(dim=-1, keepdim=True).clamp(min=1e-8)

        vis_embs[start: start + bs] = v.cpu().numpy()
        txt_embs[start: start + bs] = t.cpu().numpy()

    fused = alpha * vis_embs + (1 - alpha) * txt_embs
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    fused = fused / np.maximum(norms, 1e-8)
    return fused.astype(np.float32), ids


In [10]:
# =============================================================================
# SECTION 10 – EVALUATION METRICS
# =============================================================================
def recall_at_k(retrieved, true_id, k):
    return int(any(r == true_id for r in retrieved[:k]))

def ndcg_at_k(retrieved, true_id, k, total_rel):
    dcg  = sum(1.0 / np.log2(r+1) for r, rid in enumerate(retrieved[:k], 1) if rid == true_id)
    idcg = sum(1.0 / np.log2(r+1) for r in range(1, min(total_rel, k) + 1))
    return dcg / idcg if idcg > 0 else 0.0

def ap_at_k(retrieved, true_id, k, total_rel):
    hits, ap = 0, 0.0
    for rank, rid in enumerate(retrieved[:k], 1):
        if rid == true_id:
            hits += 1
            ap  += hits / rank
    return ap / total_rel if total_rel > 0 else 0.0

def evaluate_retrieval(gallery_embs, gallery_ids, query_embs, query_ids,
                        top_k_list, faiss_m, faiss_ef):
    dim   = gallery_embs.shape[1]
    index = faiss.IndexHNSWFlat(dim, faiss_m, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = faiss_ef
    index.hnsw.efSearch       = faiss_ef
    index.add(gallery_embs)

    max_k = max(top_k_list)
    _, I  = index.search(query_embs, max_k)

    gal_arr   = np.array(gallery_ids)
    id_counts = Counter(gallery_ids)
    metrics   = {k: {'recall': [], 'ndcg': [], 'map': []} for k in top_k_list}

    for row, qid in zip(I, query_ids):
        retrieved  = gal_arr[row].tolist()
        total_rel  = id_counts[qid]
        for k in top_k_list:
            metrics[k]['recall'].append(recall_at_k(retrieved, qid, k))
            metrics[k]['ndcg'].append(ndcg_at_k(retrieved, qid, k, total_rel))
            metrics[k]['map'].append(ap_at_k(retrieved, qid, k, total_rel))

    return {k: {'Recall@K': np.mean(metrics[k]['recall']),
                'NDCG@K':   np.mean(metrics[k]['ndcg']),
                'mAP@K':    np.mean(metrics[k]['map'])} for k in top_k_list}


def quick_recall10(model, preprocess, bbox_cache, caption_cache,
                   gallery_data, query_data, clip_dim, batch_size, device,
                   alpha=0.7):
    """Fast validation metric for checkpoint selection: Recall@10, α=0.7."""
    model.eval()
    gal_embs, gal_ids = extract_embeddings(gallery_data, model, preprocess,
                                            bbox_cache, caption_cache,
                                            clip_dim, batch_size, device, alpha,
                                            desc='Val-Gallery')
    qry_embs, qry_ids = extract_embeddings(query_data, model, preprocess,
                                            bbox_cache, caption_cache,
                                            clip_dim, batch_size, device, alpha,
                                            desc='Val-Query')
    res = evaluate_retrieval(gal_embs, gal_ids, qry_embs, qry_ids,
                              [10], FAISS_M, FAISS_EF)
    return res[10]['Recall@K']


In [ ]:
# =============================================================================
# SECTION 11 – TRAINING LOOP (main)
# =============================================================================
BEST_CKPT = OUT_DIR / 'best_clip_C_finetuned.pt'
LAST_CKPT = OUT_DIR / 'last_clip_C_finetuned.pt'

best_recall_global = -1.0
results_all_seeds  = {}

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"SEED {seed}  ({seed_idx+1}/{len(SEEDS)})")
    print(f"{'='*60}")

    # ── Setup ──────────────────────────────────────────────────────────────
    set_seed(seed)

    model, preprocess = setup_clip_for_finetuning(
        CLIP_MODEL, DEVICE, UNFREEZE_LAST_N, NUM_GPUS)
    model.to(DEVICE)
    model.train()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

    # Cosine LR schedule
    total_steps = NUM_EPOCHS * (len(train_data) // BATCH_SIZE + 1)
    scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
    scaler      = GradScaler()

    dataset = FashionTrainDataset(train_data, caption_cache, bbox_cache, preprocess)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         drop_last=True, collate_fn=collate_fn,
                         persistent_workers=(NUM_WORKERS > 0))

    raw_model = model.module if hasattr(model, 'module') else model
    seed_results = {}

    for epoch in range(1, NUM_EPOCHS + 1):
        t_ep = time.time()
        model.train()
        running_loss = 0.0
        n_batches    = 0

        for imgs, texts in tqdm(loader, desc=f'Ep {epoch}/{NUM_EPOCHS}'):
            imgs   = imgs.to(DEVICE, non_blocking=True)
            tokens = clip.tokenize(texts, truncate=True).to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast():
                img_feat = raw_model.encode_image(imgs).float()
                txt_feat = raw_model.encode_text(tokens).float()
                img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True).clamp(min=1e-8)
                txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True).clamp(min=1e-8)
                loss     = infonce_loss(img_feat, txt_feat, TEMPERATURE)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()
            n_batches    += 1

        avg_loss = running_loss / n_batches
        ep_time  = (time.time() - t_ep) / 60
        print(f"\n  Ep {epoch} | Loss: {avg_loss:.4f} | Train: {ep_time:.1f}min")

        # ── Validation (Recall@10 @ α=0.7) ─────────────────────────────────
        t_val = time.time()
        recall10 = quick_recall10(model, preprocess, bbox_cache, caption_cache,
                                   gallery_data, query_data, CLIP_DIM,
                                   BATCH_SIZE, DEVICE, alpha=0.7)
        val_time = (time.time() - t_val) / 60
        print(f"  Val Recall@10 (α=0.7): {recall10:.4f} | Val: {val_time:.1f}min")

        seed_results[epoch] = {'loss': avg_loss, 'recall10': recall10}

        # ── Save last checkpoint (always overwrite) ─────────────────────────
        ckpt = {
            'seed': seed, 'epoch': epoch, 'recall10': recall10,
            'model_state': (model.module if hasattr(model, 'module') else model).state_dict(),
            'optimizer_state': optimizer.state_dict(),
        }
        torch.save(ckpt, LAST_CKPT)
        print(f"  Saved last → {LAST_CKPT}")

        # ── Save best checkpoint (overwrite if improved) ─────────────────────
        if recall10 > best_recall_global:
            best_recall_global = recall10
            torch.save(ckpt, BEST_CKPT)
            print(f"  ★ New best Recall@10={recall10:.4f} → saved → {BEST_CKPT}")

        model.train()

    results_all_seeds[seed] = seed_results
    del model, optimizer, scheduler, scaler, loader, dataset
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\nSeed {seed} done. Best global Recall@10 so far: {best_recall_global:.4f}")



In [ ]:
# =============================================================================
# SECTION 12 – FINAL EVALUATION WITH BEST CHECKPOINT
# =============================================================================
print(f"\n{'='*60}")
print(f"FINAL EVALUATION — loading best checkpoint")
print(f"{'='*60}")

ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only = False)
print(f"Best ckpt: seed={ckpt['seed']} | epoch={ckpt['epoch']} | recall10={ckpt['recall10']:.4f}")

# Reload CLIP (fresh) and inject best weights
best_model, preprocess = clip.load(CLIP_MODEL, device=DEVICE)
best_model = best_model.float()
best_model.load_state_dict(ckpt['model_state'])
if NUM_GPUS > 1:
    best_model = nn.DataParallel(best_model)
best_model.to(DEVICE)
best_model.eval()

final_results = {}

for alpha in ALPHA_VALUES:
    print(f"\n── α = {alpha} ──")
    t0 = time.time()

    gal_embs, gal_ids = extract_embeddings(
        gallery_data, best_model, preprocess, bbox_cache, caption_cache,
        CLIP_DIM, BATCH_SIZE, DEVICE, alpha, desc=f'Gallery α={alpha}')

    qry_embs, qry_ids = extract_embeddings(
        query_data, best_model, preprocess, bbox_cache, caption_cache,
        CLIP_DIM, BATCH_SIZE, DEVICE, alpha, desc=f'Query α={alpha}')

    # Save embeddings
    np.save(OUT_DIR / f'gal_embs_C_{alpha}.npy', gal_embs)
    np.save(OUT_DIR / f'qry_embs_C_{alpha}.npy', qry_embs)
    with open(OUT_DIR / f'gal_ids_C_{alpha}.json', 'w') as f:
        json.dump(gal_ids, f)
    with open(OUT_DIR / f'qry_ids_C_{alpha}.json', 'w') as f:
        json.dump(qry_ids, f)

    # FAISS index
    dim   = gal_embs.shape[1]
    index = faiss.IndexHNSWFlat(dim, FAISS_M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = FAISS_EF
    index.hnsw.efSearch       = FAISS_EF
    index.add(gal_embs)
    faiss.write_index(index, str(OUT_DIR / f'gallery_index_C_{alpha}.faiss'))

    # Metrics
    res = evaluate_retrieval(gal_embs, gal_ids, qry_embs, qry_ids,
                              TOP_K, FAISS_M, FAISS_EF)
    final_results[str(alpha)] = res

    for k, m in res.items():
        print(f"  @{k}: Recall={m['Recall@K']:.4f}  NDCG={m['NDCG@K']:.4f}  mAP={m['mAP@K']:.4f}")

    print(f"  [{(time.time()-t0)/60:.1f}min]")

# Save results JSON
with open(OUT_DIR / 'results_config_C.json', 'w') as f:
    json.dump(final_results, f, indent=2)
print(f"\n✅  Results saved → {OUT_DIR/'results_config_C.json'}")
print(f"    Best global Recall@10 (α=0.7): {best_recall_global:.4f}")


In [17]:
# =============================================================================
# SECTION 13 – LOAD BEST MODEL & INFERENCE
# =============================================================================
print(f"\n{'='*60}")
print(f"INFERENCE MODE – Loading best model for inference")
print(f"{'='*60}")

# Input/output directory paths
INPUT_DIR = Path('/kaggle/input/datasets/venkatrrr/clip-finetuned-085')
BEST_CKPT_INPUT = INPUT_DIR / 'best_clip_C_finetuned.pt'

# Load best checkpoint with weights_only=False
ckpt = torch.load(BEST_CKPT_INPUT, map_location=DEVICE, weights_only=False)
print(f"Loaded ckpt: seed={ckpt['seed']} | epoch={ckpt['epoch']} | recall10={ckpt['recall10']:.4f}")

# Setup inference model
infer_model, infer_preprocess = clip.load(CLIP_MODEL, device=DEVICE)
infer_model = infer_model.float()
infer_model.load_state_dict(ckpt['model_state'])
if NUM_GPUS > 1:
    infer_model = nn.DataParallel(infer_model)
infer_model.to(DEVICE)
infer_model.eval()

print("✓ Model loaded and ready for inference")


# Ensure crop_with_cache is available (defined in Section 4)
if 'crop_with_cache' not in locals():
    def crop_with_cache(path: str, bbox_cache: dict) -> Image.Image:
        """Return YOLO-cropped PIL image using pre-computed bbox."""
        img = Image.open(path).convert('RGB')
        bbox = bbox_cache.get(path)
        if bbox is not None:
            x1, y1, x2, y2 = bbox
            x1, y1 = max(0, x1), max(0, y1)
            crop = img.crop((x1, y1, x2, y2))
            if crop.width > 10 and crop.height > 10:
                return crop
        return img

# Ensure bbox_cache is available (loaded in Section 4)
if 'bbox_cache' not in locals():
    print("⚠️  bbox_cache not found. Loading from file...")
    BBOX_CACHE_PATH = OUT_DIR / 'bbox_cache_C.pkl'
    if BBOX_CACHE_PATH.exists():
        with open(BBOX_CACHE_PATH, 'rb') as f:
            bbox_cache = pickle.load(f)
        print(f"✓ Loaded {len(bbox_cache):,} bbox entries")
    else:
        print(f"✗ bbox_cache file not found at {BBOX_CACHE_PATH}")
        bbox_cache = {}


@torch.no_grad()
def inference_single_image(img_path, alpha=0.6):
    """Get embedding for a single image."""
    try:
        img = crop_with_cache(img_path, bbox_cache)
        img = infer_preprocess(img).unsqueeze(0).to(DEVICE)
        
        raw_model = infer_model.module if hasattr(infer_model, 'module') else infer_model
        
        with autocast():
            v = raw_model.encode_image(img).float()
        
        v = v / v.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        vis_emb = v.cpu().numpy()
        
        text = caption_cache.get(img_path, '')
        tokens = clip.tokenize([text], truncate=True).to(DEVICE)
        
        with autocast():
            t = raw_model.encode_text(tokens).float()
        
        t = t / t.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        txt_emb = t.cpu().numpy()
        
        # Fuse
        fused = alpha * vis_emb + (1 - alpha) * txt_emb
        fused = fused / np.linalg.norm(fused, axis=1, keepdims=True)
        
        return fused[0].astype(np.float32)
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return None


@torch.no_grad()
def retrieve_top_k(query_emb, k=10, alpha=0.6):
    """Retrieve top-k gallery images for a query embedding."""
    # Load precomputed gallery embeddings from input directory if available
    try:
        gal_embs = np.load(INPUT_DIR / f'gal_embs_C_{alpha}.npy')
        with open(INPUT_DIR / f'gal_ids_C_{alpha}.json', 'r') as f:
            gal_ids = json.load(f)
        print(f"Loaded precomputed gallery embeddings (α={alpha}) from input")
    except:
        print(f"Computing gallery embeddings (α={alpha})...")
        gal_embs, gal_ids = extract_embeddings(
            gallery_data, infer_model, infer_preprocess, bbox_cache, caption_cache,
            CLIP_DIM, BATCH_SIZE, DEVICE, alpha, desc='Gallery')
    
    # Build FAISS index
    dim = gal_embs.shape[1]
    index = faiss.IndexHNSWFlat(dim, FAISS_M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = FAISS_EF
    index.hnsw.efSearch = FAISS_EF
    index.add(gal_embs)
    
    # Search
    query_emb_batch = query_emb.reshape(1, -1).astype(np.float32)
    _, I = index.search(query_emb_batch, k)
    
    retrieved_ids = [gal_ids[idx] for idx in I[0]]
    retrieved_paths = [str(p) for p, _ in gallery_data if _ in retrieved_ids][:k]
    
    return retrieved_ids[:k], retrieved_paths, gal_ids


def compute_metrics_for_query(retrieved_ids, true_id, k_list=[5, 10, 15]):
    """Compute Recall@K, NDCG@K, mAP@K for a single query."""
    # Count total relevant items in gallery with same ID
    total_rel = sum(1 for gid in gallery_data if gid[1] == true_id)
    
    metrics = {}
    for k in k_list:
        # Recall@K
        recall = int(any(rid == true_id for rid in retrieved_ids[:k]))
        
        # NDCG@K
        dcg = sum(1.0 / np.log2(r+1) for r, rid in enumerate(retrieved_ids[:k], 1) if rid == true_id)
        idcg = sum(1.0 / np.log2(r+1) for r in range(1, min(total_rel, k) + 1))
        ndcg = dcg / idcg if idcg > 0 else 0.0
        
        # mAP@K
        hits = 0
        ap = 0.0
        for rank, rid in enumerate(retrieved_ids[:k], 1):
            if rid == true_id:
                hits += 1
                ap += hits / rank
        map_k = ap / total_rel if total_rel > 0 else 0.0
        
        metrics[k] = {'Recall@K': recall, 'NDCG@K': ndcg, 'mAP@K': map_k}
    
    return metrics, total_rel


# Full evaluation on all queries
if len(query_data) > 0:
    print("\n" + "="*60)
    print("FULL MODEL EVALUATION – ALL QUERIES")
    print("="*60)
    
    for alpha in [0.6, 0.7, 0.3]:
        print(f"\n── Fusion Weight α = {alpha} ──")
        t0 = time.time()
        
        # Extract gallery embeddings
        print("Extracting gallery embeddings...")
        gal_embs, gal_ids = extract_embeddings(
            gallery_data, infer_model, infer_preprocess, bbox_cache, caption_cache,
            CLIP_DIM, BATCH_SIZE, DEVICE, alpha, desc='Gallery')
        
        # Extract query embeddings
        print("Extracting query embeddings...")
        qry_embs, qry_ids = extract_embeddings(
            query_data, infer_model, infer_preprocess, bbox_cache, caption_cache,
            CLIP_DIM, BATCH_SIZE, DEVICE, alpha, desc='Query')
        
        # Evaluate retrieval
        print("Evaluating retrieval metrics...")
        res = evaluate_retrieval(gal_embs, gal_ids, qry_embs, qry_ids,
                                  TOP_K, FAISS_M, FAISS_EF)
        
        print(f"\nResults for α = {alpha}:")
        for k in TOP_K:
            m = res[k]
            print(f"  @{k:2d}: Recall@K={m['Recall@K']:.4f}  NDCG@K={m['NDCG@K']:.4f}  mAP@K={m['mAP@K']:.4f}")
        
        elapsed = (time.time() - t0) / 60
        print(f"  Time: {elapsed:.2f} min")
    
    print(f"\n{'='*60}")
    print(f"✓ Full model evaluation complete")


INFERENCE MODE – Loading best model for inference
Loaded ckpt: seed=2023085 | epoch=1 | recall10=0.5793
✓ Model loaded and ready for inference

FULL MODEL EVALUATION – ALL QUERIES

── Fusion Weight α = 0.6 ──
Extracting gallery embeddings...


Gallery:   0%|          | 0/50 [00:00<?, ?it/s]

Extracting query embeddings...


Query:   0%|          | 0/56 [00:00<?, ?it/s]

Evaluating retrieval metrics...

Results for α = 0.6:
  @ 5: Recall@K=0.5344  NDCG@K=0.2748  mAP@K=0.1887
  @10: Recall@K=0.6133  NDCG@K=0.2835  mAP@K=0.2017
  @15: Recall@K=0.6567  NDCG@K=0.2929  mAP@K=0.2071
  Time: 2.40 min

── Fusion Weight α = 0.7 ──
Extracting gallery embeddings...


...

Evaluating retrieval metrics...

Results for α = 0.7:
  @ 5: Recall@K=0.5929  NDCG@K=0.3107  mAP@K=0.2157
  @10: Recall@K=0.6751  NDCG@K=0.3206  mAP@K=0.2305
  @15: Recall@K=0.7185  NDCG@K=0.3313  mAP@K=0.2367
  Time: 2.39 min

── Fusion Weight α = 0.3 ──
Extracting gallery embeddings...


Gallery:   0%|          | 0/50 [00:00<?, ?it/s]

Extracting query embeddings...


Query:   0%|          | 0/56 [00:00<?, ?it/s]

Evaluating retrieval metrics...

Results for α = 0.3:
  @ 5: Recall@K=0.4674  NDCG@K=0.2378  mAP@K=0.1621
  @10: Recall@K=0.5396  NDCG@K=0.2439  mAP@K=0.1724
  @15: Recall@K=0.5809  NDCG@K=0.2516  mAP@K=0.1768
  Time: 2.35 min

✓ Full model evaluation complete
